# ERP003950 (мышь) — симуляция фрагментированного 5′RACE-секвенирования через InSilicoSeq — 150bp

Имитирует реальную практику: полноразмерную V(D)J-молекулу (5′RACE-ампликон)
**нарезают** перед секвенированием на более короткие фрагменты (как в
tagmentation/sonication-библиотеках), затем каждый фрагмент читают 150bp с
двух концов на MiSeq. В отличие от `simulate_mouse_merged_insilicoseq.ipynb`
(fixed-end `--sequence_type amplicon`, читает всегда только два конца целой
молекулы), здесь используется `--sequence_type metagenomics` — каждый
`template` (полная merged-последовательность) выступает отдельным
"геномом", и `iss generate` случайно расставляет фрагменты по всей его
длине, набирая покрытие middle-участков за много ридов.

Дальше эти fastq — вход для **TRUST4** (de novo reference-guided сборка
V/D/J/C, не требует overlap между R1/R2 одной пары и не требует UMI) —
не для `AssemblePairs.py`/pRESTO, который умеет сшивать только одну пару
по overlap'у.

Модель прибора (`iss model`) обучается на реальных raw ERP003950 reads,
обрезанных `--trim-to 150` — это тот же MiSeq-профиль по циклам, что и
в fixed-end варианте; `--sequence_type` на выбор прибора не влияет,
только на расстановку ридов по референсу.

Kernel: **BCR Pipeline** (`bcr_env`), OneQ task
`bbc68913-4463-433d-b647-c3b8a76555ba`.


### 1. Env check

In [2]:
import os, sys, sysconfig, subprocess

_ENV_CANDIDATES = [
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)

print(f"Using env: {_CONDA_ENV}")
for tool in ("iss", "bowtie2", "bowtie2-build", "samtools"):
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    if not path:
        raise RuntimeError(
            f"'{tool}' not found in PATH. Install with:\n"
            f"  conda install -c bioconda -c conda-forge bowtie2 samtools -y  # (iss should already be present)"
        )
    print(tool, "->", path)
print(subprocess.run(["iss", "--version"], capture_output=True, text=True).stderr.strip())


Using env: /data/user/epishkin/conda/envs/bcr_env
iss -> /data/user/epishkin/conda/envs/bcr_env/bin/iss
bowtie2 -> /data/user/epishkin/conda/envs/bcr_env/bin/bowtie2
bowtie2-build -> /data/user/epishkin/conda/envs/bcr_env/bin/bowtie2-build
samtools -> /data/user/epishkin/conda/envs/bcr_env/bin/samtools



### 2. Config

In [3]:
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]

MERGED_FASTQ_DIR = VOLUME / "results" / DATASET / "merged" / "fastq"
RAW_FASTQ_DIR = VOLUME / "raw" / DATASET  # ERR346596_1.fastq.gz / _2.fastq.gz

# templates.fasta общий с 301bp-прогоном (сами последовательности от модели/режима не зависят,
# пересчитывать дедуп млн merged reads заново незачем). read_counts.tsv оттуда НЕ используем —
# у него другая семантика (см. ниже, шаг 4).
SHARED_TEMPLATES_DIR = VOLUME / "results" / DATASET / "simulated" / "insilicoseq" / "templates"

TARGET_READ_LENGTH = 150  # bowtie2 --trim-to обрежет raw reads до этой длины перед align'ом

# собственное дерево выхода этого прогона, рядом с 301bp (не внутри него)
OUT_BASE = VOLUME / "results" / DATASET / "simulated" / f"insilicoseq_{TARGET_READ_LENGTH}bp"
MODEL_DIR = OUT_BASE / "model"
COUNTS_DIR = OUT_BASE / "read_counts"  # coverage-based read_counts.tsv, НЕ те же файлы, что в SHARED_TEMPLATES_DIR
FASTQ_DIR = OUT_BASE / "fastq"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
for d in (MODEL_DIR, COUNTS_DIR, FASTQ_DIR, LOGS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

REF_FASTA = MODEL_DIR / "ERP003950_all_samples_reference.fasta"
BOWTIE2_INDEX = MODEL_DIR / "ERP003950_bt2_index"
BAM_PATH = MODEL_DIR / "ERP003950_real_reads_vs_merged.bam"
CUSTOM_MODEL_PREFIX = MODEL_DIR / f"ERP003950_mouse_miseq{TARGET_READ_LENGTH}"
CUSTOM_MODEL_NPZ = Path(str(CUSTOM_MODEL_PREFIX) + ".npz")

# --- ручки: тренировка модели ---
NPROC = 8
SEED = 42
COMPRESS = True
FORCE = False

# --- ручки: fragmentation-симуляция (шаги 4-5) ---
SEQUENCE_TYPE = "metagenomics"  # не "amplicon" — нужна случайная расстановка фрагментов по шаблону
TARGET_COVERAGE = 8.0  # средняя глубина покрытия на шаблон; e^-8 ~ 0.03% ожидаемой непокрытой доли
# Длина фрагмента (после "нарезки", до чтения 2x150). Пока разумная прикидка -
# поправь под реальные цифры своей библиотеки, если они известны (sonication/tagmentation insert size).
FRAGMENT_LENGTH_MEAN = 200
FRAGMENT_LENGTH_SD = 40


### 3. Train the 150bp error model

`iss model` сам не выравнивает — нужен готовый BAM реальных reads на
референс. Референс — уже собранные (`AssemblePairs.py`) merged sequences
этого же датасета: выравниваем raw R1/R2 обратно на них через `bowtie2`.
Раз реальный ERP003950 — честный 2x250, а нужна модель на 150bp — raw reads
перед выравниванием обрезаются `bowtie2 --trim-to TARGET_READ_LENGTH` (с 3'-конца,
т.е. остаётся 5'-часть рида, где качество выше). Референс и bowtie2-индекс
(шаги 3a/3b) от целевой длины рида не зависят и не пересобираются.

Тяжёлый шаг (~4.6M read pairs по всем 6 samples) — heartbeat каждые 30с.


In [5]:
import time

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(cmd)
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}); see {log_path}")
    print(f"done: elapsed={elapsed/60:.1f} min")

def iter_fastq(path):
    import gzip
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n")
            h.readline()  # +
            h.readline()  # качество
            yield seq

def iter_fasta(path):
    with open(path) as h:
        header, seq_chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_chunks)
                header, seq_chunks = line[1:], []
            else:
                seq_chunks.append(line)
        if header is not None:
            yield header, "".join(seq_chunks)


In [6]:
# 3a. Собрать один reference FASTA из merged (assemble-pass) reads всех samples
def merged_fastq_to_fasta(sample, out_handle):
    fq = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
    n = 0
    for i, seq in enumerate(iter_fastq(fq)):
        out_handle.write(f">{sample}_{i}\n{seq}\n")
        n += 1
    return n

if REF_FASTA.exists() and not FORCE:
    print(f"[skip] {REF_FASTA} exists")
else:
    total = 0
    with open(REF_FASTA, "w") as out_h:
        for sample in SAMPLES:
            n = merged_fastq_to_fasta(sample, out_h)
            total += n
            print(f"  {sample}: {n:,} reference sequences")
    print(f"wrote {REF_FASTA} ({total:,} sequences total)")


[skip] /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_all_samples_reference.fasta exists


In [7]:
# 3b. bowtie2-build
index_done_marker = Path(str(BOWTIE2_INDEX) + ".1.bt2")
if index_done_marker.exists() and not FORCE:
    print(f"[skip] bowtie2 index already built: {BOWTIE2_INDEX}")
else:
    run_with_heartbeat(
        ["bowtie2-build", "--threads", str(NPROC), str(REF_FASTA), str(BOWTIE2_INDEX)],
        LOGS_DIR / "bowtie2_build.log",
    )


[skip] bowtie2 index already built: /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_bt2_index


In [8]:
# 3c. Выровнять raw R1/R2 (все 6 samples) на merged-read референс -> отсортированный+индексированный BAM
if BAM_PATH.exists() and not FORCE:
    print(f"[skip] {BAM_PATH} exists")
else:
    r1_list = ",".join(str(RAW_FASTQ_DIR / f"{s}_1.fastq.gz") for s in SAMPLES)
    r2_list = ",".join(str(RAW_FASTQ_DIR / f"{s}_2.fastq.gz") for s in SAMPLES)
    align_cmd = (
        f"bowtie2 --local --trim-to {TARGET_READ_LENGTH} -p {NPROC} -x {BOWTIE2_INDEX} -1 {r1_list} -2 {r2_list} "
        f"2> {LOGS_DIR / 'bowtie2_align.log'} "
        f"| samtools view -bS - "
        f"| samtools sort -@ {NPROC} -o {BAM_PATH} -"
    )
    run_with_heartbeat(align_cmd, LOGS_DIR / "bowtie2_align_pipe.log", shell=True)
    subprocess.run(["samtools", "index", str(BAM_PATH)], check=True)
    print(f"indexed {BAM_PATH}")


[skip] /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_real_reads_vs_merged.bam exists


In [22]:
# 3d. iss model: построить кастомную KDE error-модель из BAM
#
# ВАЖНО: у iss/app.py есть баг — main() ловит любой AttributeError внутри
# bam.to_model() широким `except AttributeError`, печатает usage и молча
# завершается с exit code 0, даже если .npz не был записан. Поэтому:
#   - добавляем --debug, чтобы реальная причина (logger.debug(e)) попала в лог;
#   - НЕ доверяем returncode==0 — сами проверяем, что файл реально появился.
if CUSTOM_MODEL_NPZ.exists() and not FORCE:
    print(f"[skip] {CUSTOM_MODEL_NPZ} exists")
else:
    run_with_heartbeat(
        ["iss", "model", "--debug", "-b", str(BAM_PATH), "-o", str(CUSTOM_MODEL_PREFIX)],
        LOGS_DIR / "iss_model.log",
    )
    if not CUSTOM_MODEL_NPZ.exists():
        log_tail = (LOGS_DIR / "iss_model.log").read_text()[-3000:]
        raise RuntimeError(
            f"iss model exited 0 but did not write {CUSTOM_MODEL_NPZ}. "
            f"Known ISS bug: AttributeError inside bam.to_model() is swallowed "
            f"(app.py main(), bare 'except AttributeError'). Log tail:\n{log_tail}"
        )

from iss.error_models.kde import KDErrorModel
em = KDErrorModel(str(CUSTOM_MODEL_NPZ))
print(f"model: {CUSTOM_MODEL_NPZ}")
print(f"read_length = {em.read_length}")
if em.read_length != TARGET_READ_LENGTH:
    print(f"WARNING: expected read_length={TARGET_READ_LENGTH}, model actually has {em.read_length}")


[run] iss model --debug -b /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_real_reads_vs_merged.bam -o /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150
  pid=59722 log=/data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/logs/iss_model.log
  still running: pid=59722 elapsed=0.0 min
  still running: pid=59722 elapsed=0.5 min
  still running: pid=59722 elapsed=1.0 min
  still running: pid=59722 elapsed=1.5 min
  still running: pid=59722 elapsed=2.0 min
  still running: pid=59722 elapsed=2.5 min
  still running: pid=59722 elapsed=3.0 min
  still running: pid=59722 elapsed=3.5 min
  still running: pid=59722 elapsed=4.0 min
  still running: pid=59722 elapsed=4.5 min
  still running: pid=59722 elapsed=5.0 min
  still running: pid=59722 elapsed=5.5 min
  still running: pid=59722 elapsed=6.0 min
  still running: pid=59722 elapsed=6.5 min
  still running: pid=59722 elapsed=7.0 min
  still running: pid

### 4. Построить coverage-based `read_counts.tsv`

`read_counts.tsv` из 301bp-прогона хранит **реальную множественность**
(сколько раз эта точная последовательность встретилась в merged reads) —
для `amplicon`-режима (fixed-end) это правильная семантика: одна пара
ридов = одна реальная копия молекулы.

Для `metagenomics`-режима (фрагментация) семантика другая: `read_counts`
= **сколько read-пар нужно нагенерировать на этот шаблон**, чтобы набрать
покрытие всей его длины случайно расставленными фрагментами. Если взять
реальную множественность как есть — у подавляющего большинства шаблонов
(count=1, синглтоны) получится всего одна случайная пара на молекулу,
и большая часть длины останется непокрытой (см. обсуждение — при
`TARGET_COVERAGE=8` вероятность разрыва на позицию ~`e^-8`≈0.03%, а при
1 паре она огромна).

Считаем число пар отдельно на каждый шаблон, по его собственной длине:

```
N_pairs(template) = ceil(TARGET_COVERAGE * len(template) / (2 * TARGET_READ_LENGTH))
```

Пишем в **отдельный** файл (`read_counts/`), не трогая исходный
`SHARED_TEMPLATES_DIR` (он остаётся как есть для 301bp/amplicon-прогона).


In [28]:
import csv, math

def build_coverage_read_counts(sample, target_coverage=TARGET_COVERAGE,
                                read_length=TARGET_READ_LENGTH, force=FORCE):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    out_tsv = COUNTS_DIR / f"{sample}_read_counts_coverage.tsv"

    if out_tsv.exists() and not force:
        n = sum(1 for _ in open(out_tsv))
        print(f"[{sample}] [skip] {out_tsv.name} exists: {n:,} templates")
        return {"sample": sample, "status": "skipped"}

    if not templates_fa.exists():
        raise FileNotFoundError(f"Missing shared templates for {sample}: {templates_fa}")

    rows = 0
    total_pairs = 0
    lengths = []
    with open(out_tsv, "w", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        for tpl_id, seq in iter_fasta(templates_fa):
            length = len(seq)
            lengths.append(length)
            n_pairs = max(1, math.ceil(target_coverage * length / (2 * read_length)))
            writer.writerow([tpl_id, n_pairs])
            rows += 1
            total_pairs += n_pairs

    mean_len = sum(lengths) / len(lengths) if lengths else 0
    print(f"[{sample}] templates={rows:,} total_read_pairs={total_pairs:,} "
          f"mean_template_len={mean_len:.0f} min={min(lengths) if lengths else 0} max={max(lengths) if lengths else 0}")
    return {"sample": sample, "status": "built", "templates": rows, "total_read_pairs": total_pairs}


In [29]:
coverage_rows = [build_coverage_read_counts(sample) for sample in SAMPLES]


[ERR346596] [skip] ERR346596_read_counts_coverage.tsv exists: 624,945 templates
[ERR346597] [skip] ERR346597_read_counts_coverage.tsv exists: 595,793 templates
[ERR346598] [skip] ERR346598_read_counts_coverage.tsv exists: 1,302,064 templates
[ERR346599] [skip] ERR346599_read_counts_coverage.tsv exists: 713,450 templates
[ERR346600] [skip] ERR346600_read_counts_coverage.tsv exists: 552,109 templates
[ERR346601] [skip] ERR346601_read_counts_coverage.tsv exists: 689,729 templates


### 5. Run `iss generate` per sample (fragmentation/`metagenomics`, 150bp модель)

Использует `templates.fasta` из `SHARED_TEMPLATES_DIR` (те же последовательности,
что и в 301bp-прогоне — от режима не зависят), но **свой** coverage-based
`read_counts.tsv` из шага 4. `--fragment-length`/`--fragment-length-sd`
задают распределение длины фрагмента после "нарезки" (см. `FRAGMENT_LENGTH_MEAN/SD`
в конфиге — подставь реальные цифры своей библиотеки, если они известны).


In [30]:
for sample in SAMPLES:
    fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    tsv = COUNTS_DIR / f"{sample}_read_counts_coverage.tsv"
    if not fa.exists():
        raise FileNotFoundError(f"Missing shared templates for {sample}: {fa}\n"
                                 "Run step 3 in simulate_mouse_merged_insilicoseq.ipynb first.")
    if not tsv.exists():
        raise FileNotFoundError(f"Missing coverage read_counts for {sample}: {tsv}\n"
                                 "Run step 4 (build_coverage_read_counts) first.")
print("templates + coverage read_counts OK for all samples")


templates + coverage read_counts OK for all samples


In [31]:
def run_iss_generate(sample, model=str(CUSTOM_MODEL_NPZ), sequence_type=SEQUENCE_TYPE, nproc=NPROC,
                      seed=SEED, compress=COMPRESS, force=FORCE,
                      fragment_length_mean=FRAGMENT_LENGTH_MEAN, fragment_length_sd=FRAGMENT_LENGTH_SD):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    counts_tsv = COUNTS_DIR / f"{sample}_read_counts_coverage.tsv"
    out_prefix = FASTQ_DIR / sample
    ext = ".fastq.gz" if compress else ".fastq"
    r1_out = Path(str(out_prefix) + f"_R1{ext}")
    r2_out = Path(str(out_prefix) + f"_R2{ext}")

    if r1_out.exists() and r2_out.exists() and not force:
        print(f"[{sample}] [skip] {r1_out.name} exists")
        return {"sample": sample, "status": "skipped"}

    stdout_path = LOGS_DIR / f"{sample}_iss.stdout.txt"
    stderr_path = LOGS_DIR / f"{sample}_iss.stderr.txt"

    cmd = [
        "iss", "generate",
        "--genomes", str(templates_fa),
        "--readcount_file", str(counts_tsv),
        "--sequence_type", sequence_type,
        "--model", model,
        "--cpus", str(nproc),
        "--output", str(out_prefix),
    ]
    if sequence_type == "metagenomics":
        cmd += ["--fragment-length", str(fragment_length_mean), "--fragment-length-sd", str(fragment_length_sd)]
    if seed is not None:
        cmd += ["--seed", str(seed)]
    if compress:
        cmd.append("--compress")

    print(f"[{sample}] [run] {' '.join(cmd)}")
    t0 = time.time()
    with open(stdout_path, "w") as out_h, open(stderr_path, "w") as err_h:
        proc = subprocess.Popen(cmd, stdout=out_h, stderr=err_h, text=True)
        print(f"  pid={proc.pid} stdout={stdout_path.name} stderr={stderr_path.name}")
        while True:
            rc = proc.poll()
            if rc is not None:
                break
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(30)
    elapsed = time.time() - t0
    if rc != 0:
        raise RuntimeError(f"iss generate failed for {sample} (exit {rc}); see {stderr_path}")

    n_r1 = sum(1 for _ in iter_fastq(r1_out))
    print(f"[{sample}] done: R1={n_r1:,} reads elapsed={elapsed/60:.1f} min")
    return {
        "sample": sample, "status": "done", "elapsed_sec": f"{elapsed:.1f}",
        "reads_generated": n_r1, "r1_fastq": str(r1_out), "r2_fastq": str(r2_out),
    }


In [ ]:
sim_rows = [run_iss_generate(sample) for sample in SAMPLES]

qc_path = QC_DIR / "simulate_qc.tsv"
done_rows = [r for r in sim_rows if r.get("status") == "done"]
if done_rows:
    with open(qc_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(done_rows[0].keys()), delimiter="\t")
        writer.writeheader()
        writer.writerows(done_rows)
    print(f"wrote {qc_path}")

total_generated = sum(r.get("reads_generated", 0) for r in sim_rows if r.get("status") == "done")
print(f"total simulated read pairs: {total_generated:,}")


[ERR346596] [skip] ERR346596_R1.fastq.gz exists
[ERR346597] [run] iss generate --genomes /data/user/epishkin/results/ERP003950/simulated/insilicoseq/templates/ERR346597_templates.fasta --readcount_file /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/read_counts/ERR346597_read_counts_coverage.tsv --sequence_type metagenomics --model /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/model/ERP003950_mouse_miseq150.npz --cpus 8 --output /data/user/epishkin/results/ERP003950/simulated/insilicoseq_150bp/fastq/ERR346597 --fragment-length 200 --fragment-length-sd 40 --seed 42 --compress
  pid=60273 stdout=ERR346597_iss.stdout.txt stderr=ERR346597_iss.stderr.txt
  still running: pid=60273 elapsed=0.0 min
  still running: pid=60273 elapsed=0.5 min
  still running: pid=60273 elapsed=1.0 min
  still running: pid=60273 elapsed=1.5 min
  still running: pid=60273 elapsed=2.0 min
  still running: pid=60273 elapsed=2.5 min
  still running: pid=60273 elapsed=3.0 min
  s

### Notes

- Модель и BAM/index — в `results/ERP003950/simulated/insilicoseq_150bp/model/`.
- Coverage-based `read_counts.tsv` — в `.../insilicoseq_150bp/read_counts/` (свои,
  не путать с `SHARED_TEMPLATES_DIR/*_read_counts.tsv` — те для fixed-end amplicon-режима).
- Fastq-выход — `.../insilicoseq_150bp/fastq/{sample}_R1/_R2.fastq.gz`.
- Дальше — вход для TRUST4 напрямую (raw/trimmed reads, без AssemblePairs).
- Проверка покрытия vs `templates.fasta` (сколько позиций реально осталось
  непокрытыми) и оценка точности сборки TRUST4 (сравнение с ground truth
  V/J/CDR3 по каждому template) — отдельные шаги, не в этом ноутбуке.
